# TSP Walkthrough with OR-Tools

The **Travelling Salesman Problem**: given a set of locations and pairwise distances, find the shortest tour that visits every location exactly once and returns to the start.

TSP is the simplest member of the VRP family. It has one vehicle, no capacity, no time windows, no pickup/delivery constraints. Everything else in OR-Tools routing builds on top of the pieces you'll see here:

1. A **distance matrix** (or a callback that computes distances on demand).
2. A **RoutingIndexManager** — maps between problem nodes and internal solver indices.
3. A **RoutingModel** — holds the constraints and the objective.
4. **Search parameters** — *how* the solver looks for a solution.
5. A **solution object** — what the solver returns, which you walk to extract routes.

We'll build this up by hand once, then switch to the `vrp_lib` helpers and start experimenting.

## 0. Setup

We rely on autoreload so any edits to `vrp_lib` are picked up without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np

from ortools.constraint_solver import pywrapcp, routing_enums_pb2

random.seed(42)
np.random.seed(42)

## 1. Generate an instance

We'll start tiny so we can sanity-check by eye. 12 random points in a 100x100 square, with point `0` as the depot.

In [ ]:
N = 12
points = [(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(N)]

fig, ax = plt.subplots(figsize=(6, 6))
xs, ys = zip(*points)
ax.scatter(xs, ys, c="black")
ax.scatter(*points[0], c="red", s=120, label="depot")
for i, (x, y) in enumerate(points):
    ax.annotate(str(i), (x, y), textcoords="offset points", xytext=(6, 6))
ax.set_aspect("equal"); ax.legend(); ax.set_title("Random TSP instance")
plt.show()

## 2. Distance matrix

OR-Tools' routing solver works with **integer** arc costs. We compute Euclidean distances and scale up so we don't lose too much precision rounding to `int`.

If your distances naturally come from a real map (OSRM, Google Maps), you'd put those numbers here instead.

In [ ]:
SCALE = 100  # 1 unit -> 100 "distance units" so int() preserves 2 decimal places

def make_matrix(pts, scale=SCALE):
    n = len(pts)
    m = [[0] * n for _ in range(n)]
    for i, (xi, yi) in enumerate(pts):
        for j, (xj, yj) in enumerate(pts):
            if i != j:
                m[i][j] = int(round(math.hypot(xi - xj, yi - yj) * scale))
    return m

distance_matrix = make_matrix(points)
np.array(distance_matrix)

## 3. Build the routing model (by hand)

Three pieces every routing model needs:

- **`RoutingIndexManager(num_nodes, num_vehicles, depot)`** — translates between *nodes* (your problem space, 0..N-1) and internal *indices* the solver uses. They're not always the same because the solver introduces extra start/end indices per vehicle.
- **`RoutingModel(manager)`** — the model itself.
- **A transit callback** — a Python function the solver calls to ask "what does it cost to go from this index to that index?". You **register** it and then tell the model to use it as the arc cost.

In [ ]:
manager = pywrapcp.RoutingIndexManager(len(distance_matrix), 1, 0)  # 1 vehicle, depot=0
routing = pywrapcp.RoutingModel(manager)

def distance_cb(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

transit_idx = routing.RegisterTransitCallback(distance_cb)
routing.SetArcCostEvaluatorOfAllVehicles(transit_idx)

## 4. Search parameters

OR-Tools doesn't "solve" TSP optimally for free — it uses **heuristics** plus a **local search metaheuristic** to improve them. Two knobs matter most:

- **`first_solution_strategy`** — how to build the *initial* feasible tour. Options like `PATH_CHEAPEST_ARC`, `SAVINGS`, `CHRISTOFIDES`.
- **`local_search_metaheuristic`** — how to keep improving it. `GUIDED_LOCAL_SEARCH` is a strong default but needs a time limit (otherwise it stops at the first local optimum).

In [ ]:
params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.time_limit.FromSeconds(3)

solution = routing.SolveWithParameters(params)
print("Status:", routing.status())
print("Objective (scaled distance):", solution.ObjectiveValue())

## 5. Walk the solution

The solver doesn't hand you a list of nodes — you have to follow the `NextVar` chain from each vehicle's start until you hit its end. This is the same loop you'll write for every VRP variant.

In [ ]:
def extract_route(routing, manager, solution, vehicle_id=0):
    index = routing.Start(vehicle_id)
    route = []
    while not routing.IsEnd(index):
        route.append(manager.IndexToNode(index))
        index = solution.Value(routing.NextVar(index))
    route.append(manager.IndexToNode(index))
    return route

route = extract_route(routing, manager, solution)
print("Route:", " -> ".join(map(str, route)))

In [ ]:
def plot_route(points, route, title=""):
    fig, ax = plt.subplots(figsize=(6, 6))
    xs = [points[i][0] for i in route]
    ys = [points[i][1] for i in route]
    ax.plot(xs, ys, "-o")
    ax.scatter(*points[0], c="red", s=120, zorder=5, label="depot")
    for i, (x, y) in enumerate(points):
        ax.annotate(str(i), (x, y), textcoords="offset points", xytext=(6, 6))
    ax.set_aspect("equal"); ax.legend(); ax.set_title(title)
    plt.show()

plot_route(points, route, title=f"PATH_CHEAPEST_ARC + GLS, obj={solution.ObjectiveValue()}")

## 6. Same thing via `vrp_lib`

Now that you've seen the moving parts, the helpers in `vrp_lib` collapse all of the above into one call. Use this from here on so the notebook stays about experiments, not boilerplate.

In [ ]:
from vrp_lib import euclidean_distance_matrix, solve_routing

matrix = euclidean_distance_matrix(points, scale=SCALE)
result = solve_routing(matrix, num_vehicles=1, depot=0, time_limit_seconds=3)
print("Objective:", result.total_distance)
print("Route:    ", " -> ".join(map(str, result.routes[0])))

## 7. Compare first-solution strategies

Heuristic choice can matter a lot — especially before local search has had time to clean things up. We'll run several first-solution strategies with a *short* time limit so the differences show, then a longer one to see how local search closes the gap.

In [ ]:
FSS = routing_enums_pb2.FirstSolutionStrategy
strategies = {
    "PATH_CHEAPEST_ARC": FSS.PATH_CHEAPEST_ARC,
    "PATH_MOST_CONSTRAINED_ARC": FSS.PATH_MOST_CONSTRAINED_ARC,
    "SAVINGS": FSS.SAVINGS,
    "CHRISTOFIDES": FSS.CHRISTOFIDES,
    "PARALLEL_CHEAPEST_INSERTION": FSS.PARALLEL_CHEAPEST_INSERTION,
    "AUTOMATIC": FSS.AUTOMATIC,
}

rows = []
for name, strat in strategies.items():
    t0 = time.perf_counter()
    r = solve_routing(matrix, time_limit_seconds=1, first_solution_strategy=strat)
    rows.append((name, r.total_distance, round(time.perf_counter() - t0, 3)))

import pandas as pd
pd.DataFrame(rows, columns=["strategy", "objective", "wall_seconds"]).sort_values("objective")

## 8. Scaling up

12 nodes is trivial. Let's see how the same approach handles 100 nodes, and where solve time starts being interesting.

In [ ]:
rows = []
for n in [20, 50, 100, 200]:
    pts = [(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(n)]
    m = euclidean_distance_matrix(pts, scale=SCALE)
    t0 = time.perf_counter()
    r = solve_routing(m, time_limit_seconds=3)
    rows.append((n, r.total_distance, round(time.perf_counter() - t0, 3)))

pd.DataFrame(rows, columns=["n_nodes", "objective", "wall_seconds"])

## 9. Exercises

Things to try from here, in roughly increasing difficulty:

1. **Asymmetric distances.** Replace the Euclidean matrix with one where `d[i][j] != d[j][i]` (e.g. add a "wind" vector). Does the route shape change?
2. **Pin a node.** Force node 5 to be visited immediately after the depot. Hint: `routing.NextVar(routing.Start(0)).SetValues([5])`.
3. **Forbid an edge.** Make it impossible to go directly from node 3 to node 7. Hint: return a very large number from the callback for that pair.
4. **Time the local search.** Run with `time_limit_seconds` in `[1, 3, 10, 30]` on a 200-node instance and plot objective vs time.
5. **Compare to a brute baseline.** For `n <= 10`, enumerate permutations and confirm OR-Tools matches the optimum.

When any of these start producing reusable code, lift it into `src/vrp_lib/` and add a test.